In [ ]:
import os
from pathlib import Path
import pandas as pd
import geopandas as gpd
from pyproj import Transformer


FILE_2020 = Path("/gravity_index/aij_rowprob_avg_earlypeak_NZ_CL_2020.csv")
FILE_2060 = Path("/gravity_index/aij_rowprob_avg_earlypeak_NZ_CL_2060.csv")
OUTPUT_EXCEL = Path("HHY_East_West_Hub_Attraction_Analysis.xlsx")

SHP_PATH = Path("/city_shp/shi_en.shp")
PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 +x_0=0 +y_0=0 +ellps=WGS84 +datum=WGS84 +units=m +no_defs"


hhy_transformer = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)

# Transform Hu Huanyong Line coordinates (Heihe to Tengchong) into projected meters
HHY_X, HHY_Y = hhy_transformer.transform(
    [127.5, 98.5],   # Longitude: [Heihe, Tengchong]
    [50.2,  25.0],   # Latitude: [Heihe, Tengchong]
)
hhy_x0, hhy_y0 = HHY_X[0], HHY_Y[0] 
hhy_x1, hhy_y1 = HHY_X[1], HHY_Y[1]  


def judge_hhy_side(cx, cy):

    vA_x = hhy_x0 - hhy_x1
    vA_y = hhy_y0 - hhy_y1
    
    vB_x = cx - hhy_x1
    vB_y = cy - hhy_y1
    
    cross_product = vA_x * vB_y - vA_y * vB_x
    
    if cross_product <= 0:
        return 'East'
    else:
        return 'West'



def main():
    print("Step 1/5: Loading raw attraction matrices (bij)...")
    df_2020 = pd.read_csv(FILE_2020, index_col=0)
    df_2060 = pd.read_csv(FILE_2060, index_col=0)
    
    df_2020.index.name = 'Origin_City'
    df_2060.index.name = 'Origin_City'
    

    melt_2020 = df_2020.reset_index().melt(id_vars='Origin_City', var_name='Dest_City', value_name='Flow_2020')
    melt_2060 = df_2060.reset_index().melt(id_vars='Origin_City', var_name='Dest_City', value_name='Flow_2060')
    

    flow_df = pd.merge(melt_2020, melt_2060, on=['Origin_City', 'Dest_City'], how='outer').fillna(0)
    
    print(f"Step 2/5: Loading spatial shapefile layers from {SHP_PATH}...")
    gdf = gpd.read_file(SHP_PATH)
    

    if gdf.crs != PROJ_STR:
        print(" -> Dynamically reprojecting shapefile layer attributes...")
        gdf = gdf.to_crs(PROJ_STR)
        

    gdf['centroid'] = gdf.geometry.centroid
    gdf['cx'] = gdf['centroid'].x
    gdf['cy'] = gdf['centroid'].y
    
    gdf['HHY_Region_Raw'] = gdf.apply(lambda row: judge_hhy_side(row['cx'], row['cy']), axis=1)

    SHP_CITY_FIELD = "English" 
    
    if SHP_CITY_FIELD not in gdf.columns:
        raise KeyError(f"Please check why the column '{SHP_CITY_FIELD}' is missing in your shapefile attributes.")

    hhy_raw_mapping = gdf[[SHP_CITY_FIELD, 'HHY_Region_Raw']].drop_duplicates().set_index(SHP_CITY_FIELD)['HHY_Region_Raw'].to_dict()
    

    origin_full_mapping = {k: f"{v}ern Region (East of HHY Line)" if v == 'East' else f"{v}ern Region (West of HHY Line)" for k, v in hhy_raw_mapping.items()}
    flow_df['Origin_Region'] = flow_df['Origin_City'].map(origin_full_mapping).fillna('Eastern Region (East of HHY Line)')
    
    # -----------------------------------------------------------------
    # Step 3: Apply Advanced Destination Mapping (with Split Others)
    # -----------------------------------------------------------------
    print("Step 3/5: Categorizing destination hub hierarchies...")
    core_hubs = {
        # Northeastern Hubs (Beijing, Tianjin)
        'Beijing': 'Northeastern Hubs', 'Tianjin': 'Northeastern Hubs',
        # Eastern Hubs (Shanghai, Hangzhou, Nanjing)
        'Shanghai': 'Eastern Hubs', 'Hangzhou': 'Eastern Hubs', 'Nanjing': 'Eastern Hubs',
        # Central Hubs (Wuhan, Zhengzhou, Changsha)
        'Wuhan': 'Central Hubs', 'Zhengzhou': 'Central Hubs', 'Changsha': 'Central Hubs',
        # Southeastern Hubs (Guangzhou, Shenzhen)
        'Guangzhou': 'Southeastern Hubs', 'Shenzhen': 'Southeastern Hubs',
        # Western Hubs (Xian, Chongqing, Chengdu)
        'Xian': 'Western Hubs', 'Chongqing': 'Western Hubs', 'Chengdu': 'Western Hubs'
    }
    
    def classify_destination(dest_city):
        # If it's one of the 15 core hubs, return its pre-defined group
        if dest_city in core_hubs:
            return core_hubs[dest_city]
        
        # If it's an ordinary city, split into East/West via the shapefile's HHY side dictionary
        side = hhy_raw_mapping.get(dest_city, 'East') 
        return f"Others (Dispersed - {side})"

    flow_df['Dest_Hub_Type'] = flow_df['Dest_City'].apply(classify_destination)
    
    # -----------------------------------------------------------------
    # Step 4: TWO-STAGE NESTED AGGREGATION (Pure Index Version)
    # -----------------------------------------------------------------
    print("Step 4/5: Executing two-stage nested aggregation (Sum destinations, Mean cities)...")
    
    # STAGE 1 (SUM): Sum intensity index bij within each city for each target category
    city_hub_flow = flow_df.groupby(['Origin_Region', 'Origin_City', 'Dest_Hub_Type']).agg({
        'Flow_2020': 'sum',
        'Flow_2060': 'sum'
    }).reset_index()
    
    # STAGE 2 (MEAN): Compute the average index value across cities within the same region
    grouped = city_hub_flow.groupby(['Origin_Region', 'Dest_Hub_Type']).agg({
        'Flow_2020': 'mean',
        'Flow_2060': 'mean'
    }).reset_index()
    

    grouped['Absolute Change'] = grouped['Flow_2060'] - grouped['Flow_2020']
    grouped['Growth Rate (%)'] = (grouped['Absolute Change'] / grouped['Flow_2020']) * 100
    

    grouped['Flow_2020'] = grouped['Flow_2020'].round(4)
    grouped['Flow_2060'] = grouped['Flow_2060'].round(4)
    grouped['Absolute Change'] = grouped['Absolute Change'].round(4)
    grouped['Growth Rate (%)'] = grouped['Growth Rate (%)'].round(2)
    

    cols_rename = {
        'Dest_Hub_Type': 'Destination Hub Category',
        'Flow_2020': 'Mean Attraction Index (2020)',
        'Flow_2060': 'Mean Attraction Index (2060)',
        'Absolute Change': 'Absolute Change'
    }
    grouped.rename(columns=cols_rename, inplace=True)
    

    print("Step 5/5: Dividing rows and saving separate sheet indexes...")
    df_east = grouped[grouped['Origin_Region'] == 'Eastern Region (East of HHY Line)'].copy().drop(columns=['Origin_Region'])
    df_west = grouped[grouped['Origin_Region'] == 'Western Region (West of HHY Line)'].copy().drop(columns=['Origin_Region'])
    

    df_east = df_east.sort_values(by='Mean Attraction Index (2020)', ascending=False)
    df_west = df_west.sort_values(by='Mean Attraction Index (2020)', ascending=False)
    
    with pd.ExcelWriter(OUTPUT_EXCEL) as writer:
        df_east.to_excel(writer, sheet_name='Origins East of HHY Line', index=False)
        df_west.to_excel(writer, sheet_name='Origins West of HHY Line', index=False)
        
    print(f"\n Finished successfully! Clean Index tables generated at:\n{OUTPUT_EXCEL.resolve()}")


if __name__ == "__main__":
    main()

Step 1/5: Loading raw attraction matrices (bij)...
Step 2/5: Loading spatial shapefile layers from /Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp...
 -> Dynamically reprojecting shapefile layer attributes...
Step 3/5: Categorizing destination hub hierarchies...
Step 4/5: Executing two-stage nested aggregation (Sum destinations, Mean cities)...
Step 5/5: Dividing rows and saving separate sheet indexes...

🎉 Finished successfully! Clean Index tables generated at:
/Users/shirley/Desktop/plots_V2/HHY_East_West_Hub_Attraction_Analysis.xlsx
